# 🌟 Production-Level Deep Learning Pipeline
## 1️⃣ Common Utilities

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Flatten, Conv2D, MaxPooling2D, GlobalAveragePooling2D, LSTM, GRU, SimpleRNN, Embedding
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet


## 2️⃣ Image Classification Module
- 🔹 a) Custom CNN with Dense, Dropout, BatchNorm

In [ ]:
def build_custom_cnn(input_shape=(224,224,3), num_classes=2):
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(64, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(128, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


✅ Use for small to medium datasets

- 🔹 b) Transfer Learning Module

In [ ]:
def build_transfer_model(model_name='VGG16', input_shape=(224,224,3), num_classes=2, fine_tune_layers=10):
    if model_name=='VGG16':
        base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    elif model_name=='ResNet50':
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    elif model_name=='MobileNet':
        base_model = MobileNet(weights='imagenet', include_top=False, input_shape=input_shape)
    else:
        raise ValueError("Choose VGG16, ResNet50 or MobileNet")

    base_model.trainable = False  # Freeze base

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Optional fine-tuning
    for layer in base_model.layers[-fine_tune_layers:]:
        layer.trainable = True

    return model


✅ Use for medium to large datasets or real-world projects

- 🔹 c) Image Data Generator

In [ ]:
def create_image_generators(
    data_dir, target_size=(224, 224), batch_size=32, val_split=0.2
):
    datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        validation_split=val_split,
    )

    train_gen = datagen.flow_from_directory(
        data_dir,
        target_size=target_size,
        batch_size=batch_size,
        class_mode="categorical",
        subset="training",
    )

    val_gen = datagen.flow_from_directory(
        data_dir,
        target_size=target_size,
        batch_size=batch_size,
        class_mode="categorical",
        subset="validation",
    )

    return train_gen, val_gen

## 3️⃣ Sequential / Time-Series Module
- 🔹 a) RNN / LSTM / GRU Builder

In [ ]:
def build_sequential_model(
    model_type="LSTM", timesteps=10, features=1, output_dim=1, hidden_units=50
):
    model = Sequential()

    if model_type == "RNN":
        model.add(
            SimpleRNN(
                hidden_units, activation="tanh", input_shape=(timesteps, features)
            )
        )
    elif model_type == "LSTM":
        model.add(
            LSTM(hidden_units, activation="tanh", input_shape=(timesteps, features))
        )
    elif model_type == "GRU":
        model.add(
            GRU(hidden_units, activation="tanh", input_shape=(timesteps, features))
        )
    else:
        raise ValueError("Choose RNN, LSTM or GRU")

    model.add(Dense(output_dim))
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

✅ Can be used for time-series prediction or sequential classification

- 🔹 b) Text Sequential Example (Embedding + LSTM/GRU)

In [ ]:
def build_text_sequence_model(
    vocab_size=1000, seq_length=20, embedding_dim=16, model_type="LSTM"
):
    model = Sequential()
    model.add(
        Embedding(
            input_dim=vocab_size, output_dim=embedding_dim, input_length=seq_length
        )
    )

    if model_type == "LSTM":
        model.add(LSTM(32))
    elif model_type == "GRU":
        model.add(GRU(32))
    elif model_type == "RNN":
        model.add(SimpleRNN(32))

    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

✅ Can be used for NLP tasks, sentiment analysis, sequence classification

## 4️⃣ Example: Using the Pipeline

In [ ]:
# Image classification with transfer learning (ResNet50)
train_gen, val_gen = create_image_generators("data/cats_dogs")

resnet_model = build_transfer_model("ResNet50", num_classes=train_gen.num_classes)
resnet_model.fit(train_gen, validation_data=val_gen, epochs=10)

# Time-series prediction (sine wave)
import numpy as np

t = np.linspace(0, 100, 1000)
data = np.sin(0.1 * t)
seq_len = 10
X, y = [], []
for i in range(len(data) - seq_len):
    X.append(data[i : i + seq_len])
    y.append(data[i + seq_len])
X = np.array(X).reshape(-1, seq_len, 1)
y = np.array(y)

lstm_model = build_sequential_model("LSTM", timesteps=seq_len, features=1)
lstm_model.fit(X, y, epochs=20, batch_size=32)

## 🔹 5️⃣ Key Features of This Pipeline

- Modular → swap CNN, TL model, or RNN type easily  
- Supports image & sequence tasks  
- Uses Dense + Dropout + BatchNorm for better generalization  
- Transfer Learning → VGG, ResNet, MobileNet  
- RNN/LSTM/GRU → for time-series & sequential data  

This pipeline is essentially ready-to-use for real-world projects, whether you are doing:

- **Image classification** → Cats vs Dogs, Leaf Disease, MNIST  
- **Time-series prediction** → Stock prices, IoT sensor data  
- **Sequential data classification** → Text, DNA sequences


## Full Projects

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    BatchNormalization,
    Flatten,
    Conv2D,
    MaxPooling2D,
    GlobalAveragePooling2D,
    LSTM,
    GRU,
    SimpleRNN,
    Embedding,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet


def build_custom_cnn(input_shape=(224, 224, 3), num_classes=2):
    model = Sequential(
        [
            Conv2D(32, (3, 3), activation="relu", input_shape=input_shape),
            BatchNormalization(),
            MaxPooling2D(2, 2),
            Conv2D(64, (3, 3), activation="relu"),
            BatchNormalization(),
            MaxPooling2D(2, 2),
            Conv2D(128, (3, 3), activation="relu"),
            BatchNormalization(),
            MaxPooling2D(2, 2),
            Flatten(),
            Dense(256, activation="relu"),
            Dropout(0.5),
            Dense(num_classes, activation="softmax"),
        ]
    )
    model.compile(
        optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
    )
    return model


def build_transfer_model(
    model_name="VGG16", input_shape=(224, 224, 3), num_classes=2, fine_tune_layers=10
):
    if model_name == "VGG16":
        base_model = VGG16(
            weights="imagenet", include_top=False, input_shape=input_shape
        )
    elif model_name == "ResNet50":
        base_model = ResNet50(
            weights="imagenet", include_top=False, input_shape=input_shape
        )
    elif model_name == "MobileNet":
        base_model = MobileNet(
            weights="imagenet", include_top=False, input_shape=input_shape
        )
    else:
        raise ValueError("Choose VGG16, ResNet50 or MobileNet")

    base_model.trainable = False  # Freeze base

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    predictions = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(
        optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
    )

    # Optional fine-tuning
    for layer in base_model.layers[-fine_tune_layers:]:
        layer.trainable = True

    return model


def create_image_generators(
    data_dir, target_size=(224, 224), batch_size=32, val_split=0.2
):
    datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        validation_split=val_split,
    )

    train_gen = datagen.flow_from_directory(
        data_dir,
        target_size=target_size,
        batch_size=batch_size,
        class_mode="categorical",
        subset="training",
    )

    val_gen = datagen.flow_from_directory(
        data_dir,
        target_size=target_size,
        batch_size=batch_size,
        class_mode="categorical",
        subset="validation",
    )

    return train_gen, val_gen


def build_sequential_model(
    model_type="LSTM", timesteps=10, features=1, output_dim=1, hidden_units=50
):
    model = Sequential()

    if model_type == "RNN":
        model.add(
            SimpleRNN(
                hidden_units, activation="tanh", input_shape=(timesteps, features)
            )
        )
    elif model_type == "LSTM":
        model.add(
            LSTM(hidden_units, activation="tanh", input_shape=(timesteps, features))
        )
    elif model_type == "GRU":
        model.add(
            GRU(hidden_units, activation="tanh", input_shape=(timesteps, features))
        )
    else:
        raise ValueError("Choose RNN, LSTM or GRU")

    model.add(Dense(output_dim))
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


def build_text_sequence_model(
    vocab_size=1000, seq_length=20, embedding_dim=16, model_type="LSTM"
):
    model = Sequential()
    model.add(
        Embedding(
            input_dim=vocab_size, output_dim=embedding_dim, input_length=seq_length
        )
    )

    if model_type == "LSTM":
        model.add(LSTM(32))
    elif model_type == "GRU":
        model.add(GRU(32))
    elif model_type == "RNN":
        model.add(SimpleRNN(32))

    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


# Image classification with transfer learning (ResNet50)
train_gen, val_gen = create_image_generators("data/cats_dogs")

resnet_model = build_transfer_model("ResNet50", num_classes=train_gen.num_classes)
resnet_model.fit(train_gen, validation_data=val_gen, epochs=10)

# Time-series prediction (sine wave)
import numpy as np

t = np.linspace(0, 100, 1000)
data = np.sin(0.1 * t)
seq_len = 10
X, y = [], []
for i in range(len(data) - seq_len):
    X.append(data[i : i + seq_len])
    y.append(data[i + seq_len])
X = np.array(X).reshape(-1, seq_len, 1)
y = np.array(y)

lstm_model = build_sequential_model("LSTM", timesteps=seq_len, features=1)
lstm_model.fit(X, y, epochs=20, batch_size=32)